# DQ-05 · products.csv
Checks: null rate, duplicates, domain values, business rules (price, cogs, margin).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
prods = pd.read_csv('products.csv')
print(f'Shape: {prods.shape}')
prods.head(3)

## 1. Null rate

In [ ]:
null_counts = prods.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(prods)*100).round(2)}))

## 2. Duplicate product_id

In [ ]:
flag('Duplicate product_id', prods.duplicated(subset='product_id'), prods)

## 3. Domain values

In [ ]:
VALID_CAT     = {'Streetwear','Outdoor','Casual','GenZ'}
VALID_SEG     = {'Activewear','Everyday','Performance','Balanced','Standard','Premium','All-weather','Trendy'}
VALID_SIZE    = {'S','M','L','XL'}

flag('Invalid category', ~prods['category'].isin(VALID_CAT),  prods)
flag('Invalid segment',  ~prods['segment'].isin(VALID_SEG),   prods)
flag('Invalid size',     ~prods['size'].isin(VALID_SIZE),      prods)

## 4. Business rules: price, cogs

In [ ]:
flag('price ≤ 0',  prods['price'] <= 0,  prods)
flag('cogs ≤ 0',   prods['cogs']  <= 0,  prods)
flag('cogs ≥ price (negative or zero margin)', prods['cogs'] >= prods['price'], prods)

## 5. Margin distribution

In [ ]:
prods['margin'] = (prods['price'] - prods['cogs']) / prods['price']
print('Margin stats:')
print(prods['margin'].describe().round(4))
flag('Margin < 0',   prods['margin'] < 0,    prods)
flag('Margin > 0.9', prods['margin'] > 0.9,  prods)

## 6. Products không xuất hiện trong order_items

In [ ]:
items = pd.read_csv('order_items.csv', low_memory=False)
ordered_prods = set(items['product_id'])
not_ordered = ~prods['product_id'].isin(ordered_prods)
flag('Products never ordered', not_ordered, prods)
print(prods[not_ordered][['product_id','product_name','category']].head(5).to_string(index=False))

## Summary

In [ ]:
summary()